# 📗 4비트 양자화와 GPU 메모리 최적화: QLoRA 학습 설정을 측정으로 고르기

지난 시간 LoRA는 학습하는 값을 작은 어댑터로 줄였지만, 베이스 모델의 가중치는 float32 그대로 GPU에 올렸습니다. 이 교안은 베이스를 4비트로 불러오는 원리와 방법을 익히고, 학습 갱신 한 번의 GPU 메모리를 측정해 24GB GPU에서 학습할 설정을 고릅니다. 교안 02는 4비트 베이스에 어댑터를 학습하고 평가합니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="교안 01에서 배울 네 가지: 1 학습 메모리 네 가지, 2 NF4 양자화, 3 4비트로 불러와 측정하기, 4 메모리 줄이기(체크포인팅, 배치와 길이, 기울기 누적, 옵티마이저).">

**학습 목표** (그림의 번호와 같습니다)

1. **학습에 드는 GPU 메모리를 가중치, 기울기, 옵티마이저 상태, 활성값으로 나누고, 풀 파인튜닝·LoRA·QLoRA의 가중치 메모리를 계산합니다.**
2. **NF4와 블록별 absmax, 이중 양자화로 가중치가 4비트로 저장되는 과정을 실제 가중치로 계산합니다.**
3. **`BitsAndBytesConfig`로 모델을 4비트로 불러와 어댑터를 붙이고, 학습 한 번의 최대 할당 메모리와 시간을 측정합니다.**
4. **그래디언트 체크포인팅, 배치·길이, 기울기 누적, 옵티마이저로 메모리를 줄이고, 24GB GPU에서 9B 모델의 학습 설정을 고릅니다.**

## 1. 학습 메모리 네 가지와 QLoRA

학습하는 동안 GPU 메모리에는 네 가지가 올라갑니다.

- **가중치:** 모델의 파라미터 값입니다. 학습하지 않고 고정한 베이스 가중치도 순전파에 쓰이므로 GPU에 있어야 합니다.
- **기울기:** 역전파로 구한 파라미터마다의 기울기입니다. 학습하는 파라미터에만 생깁니다.
- **옵티마이저 상태:** AdamW가 학습하는 파라미터마다 두는 기울기의 1차·2차 이동평균입니다. 학습하는 파라미터에만 생깁니다.
- **활성값(activation):** 순전파의 중간 결과 가운데 역전파에 쓰려고 저장해 두는 값입니다. 파라미터 수가 아니라 한 번에 넣는 토큰 수(배치 × 길이)를 따라 커지므로, 가중치처럼 계산하지 않고 학습 갱신을 돌려 측정합니다.

<img src="images/01_training_memory.png" width="1050" alt="학습할 때 GPU 메모리는 가중치, 기울기, 옵티마이저 상태, 활성값이 차지합니다. LoRA는 기울기와 옵티마이저 상태를, QLoRA는 베이스 가중치까지 줄이지만 활성값은 줄이지 않습니다.">

**혼합 정밀도(mixed precision)**
- 계산은 16비트(bf16)로 빠르게 하고, 학습하는 가중치와 그 갱신은 fp32로 정확하게 하는 학습 방식입니다. 지난 시간 `SFTTrainer` 학습도 기본으로 이 방식이었습니다.
- 그래서 학습하는 파라미터 하나에 16바이트가 듭니다. 가중치, 기울기, AdamW 이동평균 두 개가 모두 fp32(4바이트)라 4 × 4 = 16바이트입니다.

| 방식 | 고정한 베이스 가중치 | 학습하는 값(가중치, 기울기, 옵티마이저 상태) |
|---|---|---|
| 풀 파인튜닝 | 없음(모든 파라미터를 학습) | 전체 파라미터 × 16바이트 |
| LoRA | 전체 파라미터 × 4바이트(float32, 지난 시간) 또는 × 2바이트(bf16) | 어댑터 × 16바이트 |
| QLoRA | 선형 층 × 약 0.5바이트(4비트), 나머지 층 × 2바이트(bf16) | 어댑터 × 8바이트(bf16) |

- 어댑터는 전체 파라미터의 1% 미만이라, LoRA와 QLoRA의 메모리 차이는 거의 베이스 가중치에서 납니다.
- QLoRA 어댑터만 8바이트인 것은 `SFTTrainer`가 4비트 모델의 어댑터를 bf16(2바이트)으로 바꾸기 때문입니다(5절). 네 값이 모두 2바이트라 2 × 4 = 8바이트입니다.

**QLoRA(Quantized Low-Rank Adaptation, 양자화 LoRA)**
- 베이스 모델의 선형 층 가중치를 4비트로 저장해 고정하고, 그 위에 LoRA 어댑터만 학습하는 방법입니다([Dettmers et al., 2023](https://arxiv.org/abs/2305.14314)).
- **목적:** 같은 GPU 메모리로 더 큰 모델을 학습합니다. 논문은 NF4, 이중 양자화, paged 옵티마이저 세 가지를 함께 제안했습니다.

**bitsandbytes**
- 4비트·8비트 양자화와 8비트 옵티마이저를 CUDA GPU에서 계산하는 라이브러리입니다. transformers에서는 `BitsAndBytesConfig` 설정 객체로 씁니다.

## 2. 실습 준비

### 2-1. 라이브러리와 경로

지난 시간과 같은 준비 셀입니다. 4비트로 불러오기(bitsandbytes)와 GPU 메모리 함수(`torch.cuda.*`)는 CUDA GPU에서만 동작하므로 RunPod 24GB GPU에서 실행합니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 2-2. 지난 시간과 같은 모델, 함수, 데이터

모델 이름, 토크나이저, 데이터 함수 `load_jsonl`, 노무 상담 학습 파일이 모두 지난 시간과 같습니다.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 지난 시간과 같은 토크나이저(chat template 포함). HF_TOKEN 이 없다는 Warning 은 로그인 없이 내려받는다는 안내이며 오류가 아닙니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 지난 시간과 같은 노무 상담 학습 파일(행마다 question, answer, messages)
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
print("학습 행 수:", len(train_rows))

지난 시간처럼 float32로 불러온 Base(float32 Base)입니다. 3절에서 이 모델의 실제 가중치로 4비트 양자화를 계산하고, 4-3에서 4비트로 불러온 모델과 메모리를 비교하는 기준으로 씁니다.

In [ ]:
# 지난 시간과 같은 방법(float32)으로 불러온 학습 전 모델
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,           # 허브의 모델 이름
    dtype=torch.float32,  # 지난 시간과 같은 float32
).to(device)

## 3. 4비트 양자화 원리

### 3-1. 블록별 absmax

앞 실습(양자화 단원)의 대칭 양자화는 값을 스케일로 나누고, 가장 가까운 정수로 반올림해 저장하고, 복원할 때 스케일을 곱했습니다. bitsandbytes의 4비트 양자화도 **나누고, 반올림해 저장하고, 곱해 복원하는 순서는 같습니다.** 다른 점은 나누는 값과 반올림할 눈금입니다.

| | 앞 실습의 대칭 양자화(4비트) | bitsandbytes 4비트 |
|---|---|---|
| 나누는 값(스케일) | 묶음의 최대 절댓값 ÷ 7 | 블록(64개)의 absmax 그대로 |
| 나눈 값의 범위 | -7 ~ 7 | -1 ~ 1 |
| 반올림할 눈금 | 같은 간격의 정수 15개(-7 ~ 7) | 미리 정한 16단계(위치는 3-2) |
| 저장하는 것 | 정수 | 단계 번호(0 ~ 15) |
| 복원 | 정수 × 스케일 | 단계 값 × absmax |

**absmax(absolute maximum)**
- 블록 안 값들의 가장 큰 절댓값입니다. absmax로 나누면 블록 값이 모두 -1~1 사이가 되고, 가장 큰 값은 정확히 ±1이 됩니다.

예를 들어 블록 값이 0.10, -0.42, 0.51이면 absmax는 0.51이고, 나눈 값은 0.20, -0.82, 1.00입니다. 이 값을 가장 가까운 단계로 반올림해 단계 번호 9, 1, 15를 저장합니다(NF4 기준, 3-2). 복원하면 0.16 × 0.51 ≈ 0.08처럼 원래 값과 조금 달라지고, 이 차이가 양자화 오차입니다.

<img src="images/02_blockwise_4bit.png" width="1050" alt="가중치를 64개씩 블록으로 자르고, 블록마다 absmax로 나눠 -1~1로 만든 뒤 가장 가까운 단계로 반올림해 그 번호(0부터 15)를 4비트로 저장합니다. 복원은 단계 값 × absmax입니다.">

### 3-2. NF4와 FP4의 16단계

4비트 번호는 16개뿐이므로, -1~1 안에 16단계를 어디에 놓느냐에 따라 오차가 달라집니다. bitsandbytes는 두 4비트 형식을 제공합니다.

- **NF4(NormalFloat 4-bit):** 16단계를 표준정규분포의 분위수로 정해 0 근처에 촘촘히 두고, 0을 정확히 나타내는 4비트 형식입니다([QLoRA 논문](https://arxiv.org/abs/2305.14314) 3절). 학습된 가중치는 0 근처에 몰린 정규분포 모양이라, 값이 많은 곳의 눈금을 촘촘하게 두면 같은 4비트로 오차가 줄어듭니다.
- **FP4:** 4비트를 부호 1비트, 지수 2비트, 가수 1비트로 나눈 부동소수점 형식이고, `BitsAndBytesConfig`의 기본값입니다. 저장하는 번호는 4비트라 번호 0은 `0000`, 번호 8은 `1000`입니다. FP4는 맨 앞 1비트가 부호이고, 나머지 3비트가 `000`이면 크기가 0입니다. 그래서 번호 0의 값은 +0, 번호 8의 값은 -0으로 둘 다 0이고, 번호는 16개지만 서로 다른 값은 15개입니다.

<img src="images/03_nf4_levels.png" width="1050" alt="같은 16단계라도 균등 간격은 0에 단계가 없고, NF4는 값이 많은 0 근처를 촘촘하게 나누며 0을 정확히 표현합니다.">

In [ ]:
from bitsandbytes.functional import get_4bit_type

# bitsandbytes가 쓰는 두 형식의 16단계 값(모두 -1~1)
nf4_levels = get_4bit_type(
    "nf4",          # 형식 이름
    device=device,  # 가중치와 같은 GPU에 둡니다
)
fp4_levels = get_4bit_type(
    "fp4",          # 형식 이름
    device=device,  # 가중치와 같은 GPU에 둡니다
)
levels_table = pd.DataFrame({"번호": range(16), "NF4": nf4_levels.tolist(), "FP4": fp4_levels.tolist()})
display(levels_table.round(4))

3-1의 과정(absmax로 나누기, 가장 가까운 단계로 반올림, 단계 값 × absmax로 복원)을 실제 가중치로 계산합니다. float32 Base의 첫 디코더 층 `gate_proj` 가중치(3584 × 1024, 블록 57,344개)에서 첫 블록의 앞 다섯 값으로 이 과정을 봅니다.

In [ ]:
BLOCK_SIZE = 64  # bitsandbytes 4비트 블록 크기

# gate_proj 가중치를 한 줄로 펴서 64개씩 블록으로 나눕니다. 모양 (블록 수, 64)
weight = base_model.model.layers[0].mlp.gate_proj.weight.detach()
blocks = weight.flatten().reshape(-1, BLOCK_SIZE)

# 블록마다 가장 큰 절댓값(absmax)으로 나누면 모든 값이 -1~1 사이가 됩니다
absmax = blocks.abs().max(dim=1).values
normalized = blocks / absmax.unsqueeze(1)

# 첫 블록의 값마다 NF4 16단계와의 거리를 모두 계산해, 가장 가까운 단계의 번호를 고릅니다
first_codes = (normalized[0].unsqueeze(-1) - nf4_levels).abs().argmin(dim=-1)

# 복원 값 = 번호의 단계 값 * 첫 블록의 absmax
first_block_table = pd.DataFrame({
    "원래 값": blocks[0].tolist(),
    "absmax로 나눈 값": normalized[0].tolist(),
    "NF4 번호": first_codes.tolist(),
    "복원 값": (nf4_levels[first_codes] * absmax[0]).tolist(),
})
display(first_block_table.head(5).round(4))

같은 계산을 블록 57,344개 전체에 NF4와 FP4로 해서, 원래 가중치와 복원한 가중치의 평균 절대 오차를 비교합니다.

In [ ]:
error_rows = []
for quant_type, levels in [("NF4", nf4_levels), ("FP4", fp4_levels)]:
    # 모든 값의 가장 가까운 단계 번호. 거리 모양 (블록 수, 64, 16) -> 번호 모양 (블록 수, 64)
    codes = (normalized.unsqueeze(-1) - levels).abs().argmin(dim=-1)

    # 복원 값 = 번호의 단계 값 * 그 블록의 absmax
    restored = levels[codes] * absmax.unsqueeze(1)

    # 평균 절대 오차 = 평균(|원래 값 - 복원 값|)
    error_rows.append({"형식": quant_type, "평균 절대 오차": (blocks - restored).abs().mean().item()})
error_table = pd.DataFrame(error_rows)
display(error_table.round(5))

**결과 읽기**
- **NF4와 FP4:** 평균 절대 오차가 NF4 0.00088, FP4 0.00117로 NF4가 FP4의 약 4분의 3입니다. 같은 4비트, 같은 블록이어도 16단계를 놓는 자리만으로 오차가 달라집니다.

### 3-3. 이중 양자화(double quantization)

3-1에서 가중치 64개 블록마다 absmax 하나를 함께 저장했습니다. absmax는 float32(32비트) 숫자라 이것도 메모리를 차지합니다.

**absmax가 가중치 하나에 더하는 비트: 32 / 64 = 0.5비트**
- 블록 하나에는 4비트 번호 64개와 float32 absmax 하나(32비트)가 저장됩니다.
- absmax 32비트를 가중치 64개가 나눠 쓰므로, 가중치 하나에 32 ÷ 64 = 0.5비트가 더 듭니다.
- 그래서 4비트 가중치 하나는 실제로 4 + 0.5 = 4.5비트를 씁니다.

**이중 양자화(double quantization)**
- absmax도 숫자이므로 가중치처럼 한 번 더 양자화하는 방법입니다. 가중치를 양자화하고 그 absmax를 다시 양자화하므로 '이중'입니다.
- 블록 256개에서 나온 absmax 256개를 한 묶음으로 하고, 묶음 안의 absmax를 32비트(float32) 대신 8비트로 저장합니다. 묶음마다 float32 스케일 하나를 두는데, 가중치 블록의 absmax와 같은 역할을 합니다.

<img src="images/06_double_quant.png" width="1050" alt="이중 양자화는 64개 블록마다 둔 absmax를 256개씩 묶어 8비트로 저장하고 묶음마다 fp32 스케일 하나를 둡니다. 가중치 하나의 absmax 몫이 0.5비트에서 약 0.127비트가 됩니다.">

이중 양자화를 켜면 가중치 하나에 더해지는 비트는 두 부분입니다.
- **8비트 absmax:** 가중치 64개가 나눠 쓰므로 8 ÷ 64 = 0.125비트입니다.
- **묶음 스케일(32비트):** absmax 256개, 곧 가중치 64 × 256 = 16,384개가 나눠 쓰므로 32 ÷ 16,384 ≈ 0.002비트입니다.

$\displaystyle \frac{8}{64} + \frac{32}{64 \times 256} \approx 0.127$

- absmax 몫이 0.5비트에서 약 0.127비트로 0.373비트 줄어듭니다. 그래서 4비트 가중치 하나는 4 + 0.127비트이고, 4-3절에서 이 값으로 메모리를 계산합니다.

## 4. BitsAndBytesConfig로 4비트 불러오기

### 4-1. 4비트 설정

**BitsAndBytesConfig**
- transformers에서 bitsandbytes 양자화 방식(4비트 여부, 형식, 이중 양자화, 계산 dtype)을 담는 설정 객체입니다. `from_pretrained`의 `quantization_config`로 넘깁니다.
- **기본값 함정:** `load_in_4bit=True`만 넣으면 형식은 FP4, 계산 dtype은 float32가 됩니다. 그래서 QLoRA 설정인 네 인자(`load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_use_double_quant=True`, `bnb_4bit_compute_dtype=torch.bfloat16`)를 모두 직접 넣습니다.

bf16 계산은 RTX 30xx, A5000 같은 Ampere 세대 이후 GPU에서 되고, T4 같은 옛 GPU에서는 `torch.float16`을 넣습니다.

In [ ]:
from transformers import BitsAndBytesConfig

# QLoRA 논문과 같은 4비트 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 선형 층 가중치를 4비트로 불러옵니다
    bnb_4bit_quant_type="nf4",              # 기본값 "fp4" 대신 NF4로 양자화합니다
    bnb_4bit_use_double_quant=True,         # 블록마다 둔 absmax도 8비트로 한 번 더 양자화합니다
    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산할 때 bf16으로 되돌립니다(기본값 float32는 느립니다)
)

### 4-2. 4비트로 불러오기

`quantization_config`에 위 설정을 넘기면, 불러오는 동안 transformers가 선형 층을 bitsandbytes의 `Linear4bit` 층으로 바꿉니다. 이 교안은 4비트로 불러온 학습 전 모델을 4비트 Base라고 부릅니다.

- **`Linear4bit`:** bitsandbytes의 4비트 선형 층입니다. 가중치를 4비트 번호(`torch.uint8` 한 칸에 두 개)와 블록별 absmax로 저장하고, 계산할 때마다 bf16으로 복원해 곱합니다.
- **`device_map`:** 모델의 층을 어느 장치에 둘지 정하는 딕셔너리입니다. 키는 층 이름이고 `""`는 모델 전체를 뜻합니다. 값은 장치이고, 정수는 GPU 번호입니다(`0`은 첫 번째 GPU, `cuda:0`). 그래서 `{"": 0}`은 모든 층을 첫 번째 GPU에 둡니다.
- **GPU 번호는 직접 정한 고정값입니다.** GPU가 하나인 RunPod 파드에서는 0번이 그 GPU입니다. GPU가 여러 개인 서버에서는 쓸 GPU 번호로 바꿉니다.
- **4비트 모델은 불러올 때 장치를 정합니다.** 지난 시간 float32 모델은 CPU에 불러온 뒤 `.to(device)`로 옮겼습니다. 4비트 모델은 `device_map`을 주면 가중치를 GPU에 올리면서 바로 4비트로 바꿉니다.

<img src="images/04_quantized_layers.png" width="1050" alt="BitsAndBytesConfig는 디코더 층의 선형 층(어텐션, MLP)만 4비트로 바꾸고, 임베딩, lm_head, 정규화 층은 그대로 둡니다.">

In [ ]:
# 선형 층은 4비트로, 나머지 층(임베딩, 정규화 등)은 bf16으로 불러옵니다
base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                      # 허브의 모델 이름
    quantization_config=bnb_config,  # 위 4비트 설정
    dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층의 dtype
    device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

3절에서 계산한 첫 디코더 층 `gate_proj`가 4비트로 어떻게 저장되었는지 봅니다. `quant_state`는 복원에 쓰는 정보(블록 크기, 형식, absmax)를 담은 객체입니다.

In [ ]:
# 4비트 Base 첫 디코더 층의 gate_proj 와 그 복원 정보
gate_4bit = base_4bit.model.layers[0].mlp.gate_proj
quant_state = gate_4bit.weight.quant_state

gate_4bit_info = {
    "weight dtype": str(gate_4bit.weight.dtype),
    "weight 모양": tuple(gate_4bit.weight.shape),  # 4비트 번호를 uint8 한 칸에 두 개씩 담아 한 줄로 편 모양
    "블록 크기": quant_state.blocksize,
    "형식": quant_state.quant_type,
    "absmax 개수": quant_state.absmax.numel(),      # 블록마다 하나
    "absmax dtype": str(quant_state.absmax.dtype),  # 이중 양자화로 8비트
    "임베딩 dtype": str(base_4bit.model.embed_tokens.weight.dtype),
}
display(gate_4bit_info)

**결과 읽기**
- **weight 모양:** 3584 × 1024 = 3,670,016개의 4비트 번호를 `torch.uint8` 한 칸에 두 개씩 담으면 1,835,008칸입니다. 출력의 모양 (1835008, 1)과 dtype `torch.uint8`이 이 계산과 같습니다.
- **absmax:** 블록 크기 64, 형식 `nf4`이고, 블록마다 하나인 absmax 57,344개는 이중 양자화로 `torch.uint8`에 저장되었습니다. 4비트로 바뀌지 않은 임베딩은 `torch.bfloat16`입니다.

### 4-3. 가중치 메모리: 계산과 측정

bitsandbytes는 선형 층의 가중치만 4비트로 바꾸므로, 4비트로 줄어드는 양을 계산하려면 파라미터가 선형 층에 얼마나 있는지부터 셉니다. `Qwen/Qwen3.5-0.8B`는 출력층 `lm_head`가 임베딩과 같은 텐서를 쓰고(`tie_word_embeddings`), `num_parameters()`는 이 텐서를 한 번만 셉니다.

In [ ]:
def count_params(model):
    """선형 층 가중치 수(4비트로 바뀌는 몫), 나머지 층 파라미터 수, r=8 all-linear 어댑터 파라미터 수를 딕셔너리로 돌려줍니다."""
    linear_count = 0
    adapter_count = 0
    for name, module in model.named_modules():
        # lm_head는 4비트 변환과 all-linear 어댑터에서 모두 빠집니다
        if isinstance(module, torch.nn.Linear) and name != "lm_head":
            linear_count += module.weight.numel()

            # 어댑터 파라미터 수 = r * (출력 크기 + 입력 크기), 지난 시간과 같은 식
            adapter_count += 8 * (module.out_features + module.in_features)

    # 나머지 층: 임베딩, 정규화 층, (공유하지 않으면) lm_head
    other_count = model.num_parameters() - linear_count
    return {"선형 층": linear_count, "나머지 층": other_count, "어댑터": adapter_count}


small_counts = count_params(base_model)
display(small_counts)

**결과 읽기**
- **나머지 층:** 0.8B는 임베딩(어휘 248,320 × 1024 = 254,279,680개)이 파라미터의 약 34%라, 4비트로 바뀌지 않는 몫이 큽니다. 출력의 선형 층 497,614,848개는 전체(선형 층 + 나머지 층)의 약 66%입니다.

층 종류별 파라미터 수에 방식별 파라미터 하나의 바이트(학습하는 값은 16바이트, QLoRA 어댑터는 8바이트)를 곱해 방식별 가중치 메모리를 계산하고, 실제로 불러온 두 모델의 모델 메모리 옆에 둡니다.

**모델 메모리(`get_memory_footprint`)**
- 모델의 파라미터와 버퍼가 차지하는 바이트의 합입니다. absmax 같은 복원 정보는 파라미터가 아니라서 들어가지 않습니다.

In [ ]:
# 4비트 가중치 하나의 바이트: 4비트 번호 + 이중 양자화한 absmax 몫 0.127비트
nf4_bytes = (4 + 0.127) / 8


def weight_table(counts):
    """풀 파인튜닝, LoRA(float32 베이스, bf16 베이스), QLoRA의 가중치 메모리 계산값(GB) 테이블을 돌려줍니다."""
    total = counts["선형 층"] + counts["나머지 층"]
    adapter = counts["어댑터"]

    # QLoRA 베이스: 선형 층은 4비트, 4비트가 아닌 층(임베딩, 정규화 등)은 bf16 2바이트
    qlora_base = counts["선형 층"] * nf4_bytes + counts["나머지 층"] * 2

    # (방식, 고정한 베이스 가중치 바이트, 학습 파라미터 수, 학습 파라미터 하나의 바이트)
    # 학습하는 값: 가중치, 기울기, 1차, 2차 이동평균 4개가 fp32면 4바이트씩 16바이트, QLoRA 어댑터는 bf16이라 2바이트씩 8바이트
    methods = [
        ("풀 파인튜닝", 0, total, 16),
        ("LoRA(float32 베이스)", total * 4, adapter, 16),
        ("LoRA(bf16 베이스)", total * 2, adapter, 16),
        ("QLoRA", qlora_base, adapter, 8),
    ]
    rows = []
    for method, base_bytes, trainable_count, bytes_per_param in methods:
        trainable_bytes = trainable_count * bytes_per_param
        rows.append({
            "방식": method,
            "베이스 가중치(GB)": round(base_bytes / 1024 ** 3, 2),
            "학습하는 값(GB)": round(trainable_bytes / 1024 ** 3, 2),
            "합계(GB)": round((base_bytes + trainable_bytes) / 1024 ** 3, 2),
        })
    return pd.DataFrame(rows)


# 0.8B 계산값
small_weight_table = weight_table(small_counts)

# 측정값: 불러온 두 모델의 모델 메모리(GB). 불러오지 않은 방식은 None(테이블에는 NaN)
float32_gb = round(base_model.get_memory_footprint() / 1024 ** 3, 2)
nf4_gb = round(base_4bit.get_memory_footprint() / 1024 ** 3, 2)
small_weight_table["모델 메모리 측정(GB)"] = [None, float32_gb, None, nf4_gb]
display(small_weight_table)

**결과 읽기**
- **계산과 측정:** 측정한 모델 메모리(float32 Base 2.80GB, 4비트 Base 0.71GB)가 베이스 가중치 계산값과 같습니다.
- **줄어드는 비율:** bf16 베이스 약 1.4GB가 4비트에서 약 0.71GB로 절반 정도만 줄어듭니다. 선형 층이 4분의 1이 되어도 bf16 그대로인 임베딩이 남기 때문입니다.

float32 Base는 더 쓰지 않으므로 GPU 메모리에서 지웁니다.

In [ ]:
import gc

# float32 Base와 그 가중치에서 만든 텐서를 지웁니다(한 번만 실행)
del base_model, weight, blocks, absmax, normalized, codes, restored
gc.collect()              # 더 이상 쓰지 않는 객체를 메모리에서 정리합니다
torch.cuda.empty_cache()  # PyTorch가 잡아 두었던 빈 GPU 메모리를 돌려줍니다

## 5. 4비트 Base에 LoRA 어댑터 붙이기

4비트 Base는 고정한 채, 그 위에 지난 시간과 같은 LoRA 어댑터를 붙여 학습합니다. 기울기와 옵티마이저 상태는 어댑터에만 생깁니다.

<img src="images/05_qlora_structure.png" width="1050" alt="QLoRA: 4비트로 고정한 W는 계산할 때만 복원하고, 옆에 붙인 LoRA 어댑터 A, B만 학습합니다. 기울기와 옵티마이저 상태는 어댑터에만 생깁니다.">

### 5-1. SFTTrainer가 4비트 모델에 하는 일

교안 02는 4비트 Base와 어댑터 설정(`peft_config=lora_config`)을 함께 `SFTTrainer`에 넘깁니다. `SFTTrainer`는 학습을 시작하기 전에 모델을 이렇게 바꿉니다(trl 1.16 기준).

1. `get_peft_model`로 어댑터를 붙입니다. 베이스 파라미터는 모두 고정되고(`requires_grad=False`) 어댑터만 학습합니다.
2. 4비트 모델이면 학습할 어댑터를 bf16으로 바꿉니다(QLoRA 논문 권고). 4비트로 바뀌지 않은 임베딩과 정규화 층은 bf16 그대로이고, 층 사이를 흐르는 값도 bf16입니다.

**`prepare_model_for_kbit_training`은 부르지 않습니다.** peft 문서가 안내하는 이 학습 준비 함수는 4비트가 아닌 층을 fp32로 올려 활성값 메모리를 늘립니다(강사 실행에서 0.8B QLoRA 최대 할당 메모리 16.59GB -> 19.13GB). trl `SFTTrainer`도 이 함수를 부르지 않습니다.

### 5-2. LoRA 어댑터 붙이기

지난 시간 본실습과 글자까지 같은 `lora_config`로, `SFTTrainer`가 하는 두 가지를 4비트 Base에 직접 해 봅니다.

In [ ]:
from peft import LoraConfig

# 지난 시간 본실습과 같은 설정: 출력층을 뺀 모든 선형 층에 r=8 어댑터, 배율 16 / 8 = 2.0
lora_config = LoraConfig(
    r=8,                          # B, A 사이의 크기(rank)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # 어댑터 입력에 거는 드롭아웃 비율
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

In [ ]:
from peft import get_peft_model

# 4비트 Base의 선형 층(Linear4bit)마다 어댑터를 붙입니다. 베이스 파라미터는 모두 고정됩니다
qlora_model = get_peft_model(
    base_4bit,    # 4비트로 불러온 Base
    lora_config,  # 지난 시간과 같은 LoRA 설정
)

# SFTTrainer 처럼 학습할 어댑터만 bf16으로 바꿉니다(어댑터는 fp32로 만들어집니다)
for param in qlora_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)
qlora_model.print_trainable_parameters()

**결과 읽기**
- **trainable params:** 5,411,328개입니다. 4비트는 베이스만 바꾸므로 학습 파라미터 수가 지난 시간 LoRA와 같고, 불러온 직후 학습 상태였던 임베딩과 정규화 층은 `get_peft_model`이 고정했습니다.

In [ ]:
# 첫 디코더 층 gate_proj: 4비트 층(base_layer) 옆에 어댑터 lora_A, lora_B가 붙었습니다
qlora_gate = qlora_model.base_model.model.model.layers[0].mlp.gate_proj
gate_parts = {
    "base_layer": type(qlora_gate.base_layer).__name__,
    "lora_A 모양": tuple(qlora_gate.lora_A["default"].weight.shape),
    "lora_B 모양": tuple(qlora_gate.lora_B["default"].weight.shape),
    "어댑터 dtype": str(qlora_gate.lora_A["default"].weight.dtype),
}
display(gate_parts)

**결과 읽기**
- **`base_layer`:** 4-2와 같은 `Linear4bit`입니다. 지난 시간에는 `Linear`였던 자리이고, 어댑터 `lora_A`, `lora_B`의 모양은 지난 시간과 같습니다.
- **어댑터 dtype:** `torch.bfloat16`입니다. 교안 02에서 `SFTTrainer`가 학습하는 모델과 같은 모양입니다.

어댑터를 확인했으므로 모델을 지웁니다. 측정은 설정마다 모델을 새로 불러 합니다.

In [ ]:
# 이 절의 모델과 모델 안을 가리키는 변수를 지웁니다(한 번만 실행)
del qlora_model, base_4bit, qlora_gate, gate_4bit, quant_state, param
gc.collect()
torch.cuda.empty_cache()

## 6. 학습 메모리 측정

### 6-1. 측정 함수

기울기, 옵티마이저 상태, 활성값은 학습 설정에 따라 달라지므로, 어떤 설정이 GPU에 들어가는지는 학습 갱신을 몇 번 돌려 측정해 확인합니다.

**최대 할당 메모리**
- 측정 구간에서 텐서가 GPU 메모리를 가장 많이 차지한 순간의 양입니다(`torch.cuda.max_memory_allocated()`).
- **목적:** 이 값이 GPU 메모리를 넘으면 학습이 `CUDA out of memory`(메모리 부족, OOM) 에러로 멈추므로, 설정마다 이 값을 비교합니다.
- **예약 메모리(`torch.cuda.max_memory_reserved()`):** PyTorch가 텐서를 담으려고 GPU에서 잡아 둔 양의 최대치입니다. 할당 메모리보다 크거나 같고, `nvidia-smi`에 보이는 값에 가깝습니다.

<img src="images/07_peak_memory_timeline.png" width="1050" alt="GPU 메모리는 갱신마다 순전파에서 활성값만큼 올랐다가 내려옵니다. 옵티마이저 상태가 생긴 준비 갱신 뒤에 최대치 기록을 다시 잡고, 예약 메모리는 할당 메모리 위에 남습니다.">

| 함수 | 하는 일 |
|---|---|
| `torch.cuda.reset_peak_memory_stats()` | 최대치 기록을 지금 값으로 다시 잡습니다. 측정 구간의 시작에 부릅니다 |
| `torch.cuda.empty_cache()` | 쓰지 않는 캐시만 GPU에 돌려줍니다. 살아 있는 텐서는 지우지 못합니다 |
| `torch.cuda.synchronize()` | GPU 계산이 끝날 때까지 기다립니다. GPU는 CPU와 따로 돌므로 시간을 측정하기 전후에 부릅니다 |

측정할 입력은 노무 학습 데이터 전체를 chat template 문자열로 바꿔 토큰 번호를 한 줄로 이어 붙인 토큰 열입니다. 배치 크기와 길이를 원하는 값으로 바꾸려고 이어 붙인 열에서 잘라 씁니다(측정은 메모리와 시간만 보므로 예시 경계는 상관없습니다).

In [ ]:
# 학습 데이터의 대화(messages)마다 chat template 문자열의 토큰 수를 세고, 그 토큰 번호를 한 리스트에 이어 붙입니다
labor_lengths = []
labor_ids = []
for row in train_rows:
    chat_text = tokenizer.apply_chat_template(
        row["messages"],  # system, user, assistant 메시지
        tokenize=False,   # 토큰 번호가 아니라 문자열로 받습니다
    )
    row_ids = tokenizer(chat_text)["input_ids"]
    labor_lengths.append(len(row_ids))
    labor_ids.extend(row_ids)

# 1차원 정수 텐서(LongTensor). 모델 입력은 이 자료형이어야 합니다
token_ids = torch.tensor(labor_ids)

# 가장 긴 예시의 토큰 수. 설정을 고를 때 길이로 씁니다
labor_max_length = max(labor_lengths)
print("가장 긴 예시(토큰):", labor_max_length)

`load_lora_model(model_name, load_mode, gradient_checkpointing)`은 베이스를 `load_mode`대로 불러와 같은 `lora_config` 어댑터를 붙여 돌려줍니다. `load_mode`는 `"float32"`(지난 시간 LoRA), `"bf16"`(bf16 LoRA), `"4bit"`(QLoRA) 가운데 하나입니다. `"4bit"`일 때는 5-1의 `SFTTrainer`처럼 어댑터를 bf16으로 바꿉니다.

`device_map`은 4-2처럼 `{"": 0}`으로 모델 전체를 GPU 0번에 둡니다. `"auto"`를 쓰면 GPU가 모자랄 때 일부 층이 CPU로 옮겨져 측정이 엉뚱하게 느려집니다.

In [ ]:
def load_lora_model(model_name, load_mode, gradient_checkpointing):
    """베이스 모델을 load_mode("float32", "bf16", "4bit")로 불러와 lora_config 어댑터를 붙인 모델을 돌려줍니다(SFTTrainer 에 peft_config 를 넘긴 것과 같게)."""
    if load_mode == "4bit":
        # QLoRA: 선형 층을 4비트로 불러옵니다. 4비트로 바뀌지 않는 층(임베딩, 정규화)은 bf16 그대로 둡니다
        model = AutoModelForCausalLM.from_pretrained(
            model_name,                      # 허브의 모델 이름
            quantization_config=bnb_config,  # 선형 층을 NF4 4비트로 불러오는 설정
            dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층(임베딩, 정규화)의 dtype
            device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
        )
    else:
        # LoRA: float32(지난 시간과 같음) 또는 bf16으로 불러옵니다
        if load_mode == "bf16":
            dtype = torch.bfloat16
        else:
            dtype = torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,          # 허브의 모델 이름
            dtype=dtype,         # 가중치를 둘 dtype(float32 또는 bf16)
            device_map={"": 0},  # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
        )
    if gradient_checkpointing:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},  # PyTorch 권장 체크포인팅 방식
        )

    # 세 방식 모두 같은 lora_config 로 어댑터를 붙입니다(베이스 가중치는 고정됩니다)
    model = get_peft_model(model, lora_config)
    if gradient_checkpointing:
        model.enable_input_require_grads()  # 체크포인팅을 켜면 SFTTrainer 처럼 입력 쪽 기울기 길을 열어 둡니다

    # 4비트 모델은 SFTTrainer 처럼 학습할 어댑터를 bf16으로 바꿉니다(QLoRA 논문 권고)
    if load_mode == "4bit":
        for param in model.parameters():
            if param.requires_grad:
                param.data = param.data.to(torch.bfloat16)
    return model

`measure_train_step`은 학습 루프를 직접 돌며 측정합니다.

- **혼합 정밀도:** `torch.autocast(..., dtype=torch.bfloat16)` 안에서 순전파합니다. 지난 시간 `SFTTrainer`가 기본으로 쓰던 혼합 정밀도(1절)와 같은 계산입니다.
- **준비 갱신:** AdamW의 옵티마이저 상태는 첫 `optimizer.step()` 때 만들어지므로, 첫 갱신 뒤에 최대치 기록과 시작 시각을 다시 잡고 나머지 `steps`번을 측정합니다.
- **로짓 하한:** 로짓(logit)은 토큰마다 어휘 248,320개에 매긴 다음 토큰 점수입니다. 손실을 계산할 때 로짓의 fp32 사본을 만들므로, 로짓 메모리는 적어도 배치 × 길이 × 248,320 × 4바이트입니다. 이 값을 로짓 하한이라 부릅니다(실제로는 bf16 로짓 등이 더 붙어 이보다 큽니다).
- **`SFTTrainer`와의 차이:** trl 1.16 `SFTTrainer`는 기본 설정에서 로짓 전체를 한꺼번에 만들지 않고 손실과 묶어 계산합니다. 그래서 이 함수로 잰 메모리는 실제 `SFTTrainer` 학습보다 큰 쪽(상한)입니다.

In [ ]:
import time


def measure_train_step(model, optimizer, token_ids, batch_size, seq_len, accumulation_steps=1, steps=3):
    """준비 갱신 1번 뒤 steps번 갱신하며 최대 할당 메모리, 예약 메모리(GB), 갱신당 시간(초)을 딕셔너리로 돌려줍니다."""
    # 앞 측정이 잡아 둔 캐시를 돌려줘, 예약 메모리를 이 설정만으로 측정합니다
    gc.collect()
    torch.cuda.empty_cache()

    # 이어 붙인 토큰 열 앞에서부터 batch_size * seq_len개를 잘라 (배치, 길이) 모양으로 만듭니다
    batch = token_ids[:batch_size * seq_len].reshape(batch_size, seq_len).to(device)

    # 첫 회(step 0)는 옵티마이저 상태를 만드는 준비 갱신이고, 나머지 steps번이 측정 구간입니다
    model.train()
    for step in range(steps + 1):
        optimizer.zero_grad()

        # 같은 배치를 accumulation_steps번 돌려 기울기를 .grad에 더합니다(메모리와 시간만 보므로 배치 내용은 같아도 됩니다)
        for _ in range(accumulation_steps):
            # 혼합 정밀도: SFTTrainer가 기본으로 쓰는 것과 같은 계산(autocast)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(
                    input_ids=batch,   # 입력 토큰
                    labels=batch,      # 다음 토큰 예측의 정답(모델이 한 칸 밀어 손실을 계산합니다)
                    use_cache=False,   # 생성용 캐시를 만들지 않습니다(학습과 같은 조건)
                ).loss

            # 누적 평균 = 손실 / 누적 횟수 의 기울기를 더합니다
            (loss / accumulation_steps).backward()

        optimizer.step()

        # 준비 갱신이 끝나면 최대 메모리 기록과 시작 시각을 다시 잡습니다
        if step == 0:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()

    # GPU 계산이 모두 끝난 뒤 시간을 계산합니다
    torch.cuda.synchronize()
    seconds_per_update = (time.perf_counter() - start) / steps
    return {
        "배치 크기": batch_size,
        "시퀀스 길이": seq_len,
        "누적 횟수": accumulation_steps,
        "유효 배치": batch_size * accumulation_steps,
        "최대 할당(GB)": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
        "예약(GB)": round(torch.cuda.max_memory_reserved() / 1024 ** 3, 2),
        "갱신당 시간(초)": round(seconds_per_update, 3),
    }

### 6-2. float32 LoRA, bf16 LoRA, QLoRA 비교

세 방식을 같은 배치 크기 4, 시퀀스 길이 512, 누적 1, 그래디언트 체크포인팅 끔으로 측정합니다. 길이는 로짓과 층 활성값 몫이 잘 보이도록 가장 긴 예시(`labor_max_length`)보다 길게 잡았습니다. 학습 파라미터 수는 세 방식이 같으므로 차이는 베이스 가중치와 그 계산에서 옵니다.

In [ ]:
# 측정 테이블에서 볼 열. 배치 크기, 길이, 누적 횟수는 모든 행이 같아 빼고 봅니다
measure_columns = ["최대 할당(GB)", "예약(GB)", "갱신당 시간(초)"]

# 세 방식을 차례로 불러와 같은 조건으로 측정합니다
# causal_conv1d, flash-linear-attention 이 없다는 안내는 빠른 커널 대신 PyTorch 기본 구현으로 계산한다는 뜻이며 오류가 아닙니다
load_results = {}
for load_mode in ["float32", "bf16", "4bit"]:
    model = load_lora_model(
        MODEL_NAME,                    # 0.8B 모델
        load_mode,                     # 베이스를 불러오는 방식
        gradient_checkpointing=False,  # 지난 시간 LoRA와 같은 조건(끔)
    )

    # 고정한 파라미터는 기울기가 없어 옵티마이저가 건너뛰므로, 옵티마이저 상태는 어댑터에만 생깁니다
    optimizer = torch.optim.AdamW(
        model.parameters(),  # 모든 파라미터(기울기가 있는 어댑터만 갱신됩니다)
        lr=2e-4,             # 지난 시간 LoRA와 같은 학습률
    )
    load_results[load_mode] = measure_train_step(
        model,         # 측정할 모델
        optimizer,     # 측정할 옵티마이저
        token_ids,     # 노무 토큰 열
        batch_size=4,  # 배치 크기
        seq_len=512,   # 시퀀스 길이
    )

    # 다음 방식을 같은 조건에서 측정하도록 모델과 옵티마이저를 지웁니다
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

load_table = pd.DataFrame.from_dict(
    load_results,    # 방식 이름 -> 측정 딕셔너리
    orient="index",  # 방식 이름을 행 이름으로
)
display(load_table[measure_columns])

**결과 읽기**
- **최대 할당 메모리:** 강사 실행에서 float32 20.28GB, bf16 17.31GB, 4비트 16.59GB입니다. bf16과 4비트의 차이(0.72GB)는 가중치 차이와 거의 같습니다.
- **가중치보다 큰 몫:** 배치 4 × 길이 512의 로짓 하한만 약 1.9GB로, bf16과 4비트의 가중치 차이(약 0.7GB)보다 큽니다. 0.8B 모델에서는 최대 할당 메모리를 가중치보다 활성값과 로짓이 더 크게 좌우하므로, 4비트로 아끼는 몫이 전체의 5%도 되지 않습니다. 4비트의 효과는 가중치가 큰 9B(8절)에서 커집니다.
- **갱신당 시간:** float32 0.474초, bf16 0.456초, 4비트 0.503초로 4비트가 bf16보다 약 10% 느립니다. 4비트 가중치는 계산할 때마다 bf16으로 복원하므로, 4비트는 가중치 메모리를 아끼는 대신 갱신 시간을 조금 더 냅니다.

## 7. 메모리 줄이기

### 7-1. 그래디언트 체크포인팅

**그래디언트 체크포인팅(gradient checkpointing)**
- 순전파 때 층마다 입력만 저장하고, 층 안의 중간 결과는 역전파 때 그 층을 다시 계산해 얻는 기법입니다. transformers는 모든 디코더 층에 이 방식을 겁니다.
- **목적:** 층 안의 활성값을 저장하지 않아 메모리를 줄입니다. 대가는 순전파를 한 번 더 하는 시간입니다(`SFTConfig`가 물려받는 [`TrainingArguments` 설명](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments.gradient_checkpointing)은 약 20% 느려진다고 안내합니다).

<img src="images/11_gradient_checkpointing.png" width="1050" alt="그래디언트 체크포인팅은 층 안의 중간 결과 대신 층 입력만 저장했다가 역전파 때 그 층을 다시 계산합니다. 메모리는 줄고 시간은 늘어납니다.">

**켜고 끄는 곳은 두 곳입니다.**
- **모델:** `model.gradient_checkpointing_enable()`. `load_lora_model`이 `gradient_checkpointing=True`일 때 부릅니다.
- **학습 설정:** `SFTConfig(gradient_checkpointing=..., gradient_checkpointing_kwargs={"use_reentrant": False})`. trl `SFTConfig`의 기본값이 `True`이고, 지난 시간까지는 조건을 맞추려고 `False`로 껐습니다. 모델만 끄면 학습을 시작할 때 다시 켜집니다.

체크포인팅을 켠 QLoRA를 같은 배치 4, 길이 512로 측정해 끈 측정(`load_results["4bit"]`)과 비교합니다. 이 모델과 옵티마이저는 기울기 누적까지 계속 씁니다.

In [ ]:
# 체크포인팅을 켠 QLoRA 모델과 옵티마이저
model = load_lora_model(
    MODEL_NAME,                   # 0.8B 모델
    "4bit",                       # QLoRA
    gradient_checkpointing=True,  # 체크포인팅을 켭니다
)
optimizer = torch.optim.AdamW(
    model.parameters(),  # 기울기가 있는 어댑터만 갱신됩니다
    lr=2e-4,             # 같은 학습률
)

# 끔은 4비트 측정을 그대로 쓰고, 켬만 새로 측정합니다
checkpointing_results = {
    "끔": load_results["4bit"],
    "켬": measure_train_step(
        model,         # 체크포인팅을 켠 QLoRA
        optimizer,     # 같은 AdamW
        token_ids,     # 노무 토큰 열
        batch_size=4,  # 끔 측정과 같은 배치 크기
        seq_len=512,   # 끔 측정과 같은 길이
    ),
}
checkpointing_table = pd.DataFrame.from_dict(
    checkpointing_results,  # 끔, 켬 -> 측정 딕셔너리
    orient="index",         # 끔, 켬을 행 이름으로
)
display(checkpointing_table[measure_columns])

**결과 읽기**
- **메모리와 시간:** 강사 실행에서 켬의 최대 할당 메모리는 6.55GB로 끔 16.59GB보다 10.04GB(약 60%) 작고, 갱신당 시간은 0.503초에서 0.752초로 약 1.5배입니다. 느려지는 정도는 모델과 GPU에 따라 다르므로(문서는 약 20%), 학습할 환경에서 측정합니다.

### 7-2. 배치 크기와 시퀀스 길이

활성값은 한 번에 넣는 토큰 수(배치 크기 × 시퀀스 길이)를 따라 커집니다. 언어 모델에서는 층 활성값과 로짓을 나눠 봅니다.

- **층 활성값:** 배치 × 길이 × 은닉 크기 × 층 수를 따라 커지고, 체크포인팅으로 줄어듭니다.
- **로짓:** 배치 × 길이 × 어휘 248,320을 따라 커지고, 체크포인팅과 상관없이 남습니다.

<img src="images/12_activation_size.png" width="1050" alt="활성값은 배치 크기와 시퀀스 길이에 비례해 커집니다. 층마다 저장하는 값에 더해 어휘 248,320개짜리 로짓도 배치 × 길이만큼 쌓입니다.">

7-1의 체크포인팅 켬 측정(배치 4 × 길이 512 = 2,048토큰)에 세 설정을 더해 봅니다.

- 토큰 수가 절반인 두 설정: 2 × 512, 4 × 256(둘 다 1,024토큰)
- 토큰 수가 두 배인 설정: 4 × 1024(4,096토큰)

In [ ]:
# 체크포인팅 켬 측정에 배치나 길이만 바꾼 세 설정을 더합니다
shape_results = [checkpointing_results["켬"]]
for batch_size, seq_len in [(2, 512), (4, 256), (4, 1024)]:
    shape_results.append(measure_train_step(
        model,                  # 체크포인팅을 켠 QLoRA
        optimizer,              # 같은 AdamW
        token_ids,              # 노무 토큰 열
        batch_size=batch_size,  # 바꾼 배치 크기
        seq_len=seq_len,        # 바꾼 길이
    ))

# 활성값은 토큰 수를 따르므로 토큰 수 열을 더해 그 순서로 봅니다. 누적 횟수와 유효 배치 열은 빼고 봅니다
activation_table = pd.DataFrame(shape_results)
activation_table["토큰 수"] = activation_table["배치 크기"] * activation_table["시퀀스 길이"]
activation_table = activation_table.sort_values("토큰 수")
display(activation_table[["배치 크기", "시퀀스 길이", "토큰 수"] + measure_columns])

**결과 읽기**
- **토큰 수가 같은 설정:** 강사 실행에서 2 × 512와 4 × 256(둘 다 1,024토큰)의 최대 할당 메모리가 모두 3.66GB입니다. 메모리는 배치와 길이를 어떻게 나누든 토큰 수를 따릅니다.
- **토큰 수에 따른 증가:** 토큰 수가 1,024, 2,048, 4,096으로 늘 때 최대 할당 메모리는 3.66GB, 6.55GB, 12.34GB로, 토큰 1,024개마다 약 2.9GB씩 늘었습니다. 체크포인팅을 켰으므로 늘어난 양은 대부분 로짓에서 나옵니다.

**`max_length`는 데이터의 가장 긴 예시(노무 데이터는 `labor_max_length`토큰)로 정합니다.** `SFTTrainer`는 배치마다 그 배치에서 가장 긴 예시까지만 패딩을 채우므로, `max_length`를 가장 긴 예시보다 길게 잡아도 메모리를 더 쓰지 않습니다. `max_length`를 긴 예시보다 짧게 잡으면 긴 예시가 잘려 그 배치의 활성값이 줄지만, 잘린 뒷부분은 학습에서 빠집니다.

### 7-3. 기울기 누적

배치 크기를 줄이면 메모리는 줄지만 갱신 한 번에 쓰는 예시 수도 줄어 학습이 달라집니다.

**기울기 누적(gradient accumulation)**
- 작은 배치마다 `(손실 / 누적 횟수).backward()`로 기울기를 `.grad`에 더하고, 정한 횟수마다 한 번 `optimizer.step()`으로 갱신하는 기법입니다.
- **목적:** 메모리는 작은 배치만큼만 쓰면서, 큰 배치로 갱신한 것과 같은 평균 기울기로 갱신합니다.
- **유효 배치:** 갱신 한 번에 쓰인 예시 수입니다. 유효 배치 = GPU당 배치 크기(`per_device_train_batch_size`) × 누적 횟수(`gradient_accumulation_steps`) × GPU 수입니다.

<img src="images/13_gradient_accumulation.png" width="1050" alt="배치 4의 기울기를 4번 더해 한 번 갱신하면 배치 16으로 한 번 갱신한 것과 같습니다(유효 배치 = 배치 크기 × 누적 횟수).">

유효 배치 16을 배치 16 × 누적 1과 배치 4 × 누적 4로 만들어, 길이 `labor_max_length`로 측정합니다.

In [ ]:
# (배치 크기, 누적 횟수) 두 설정 모두 유효 배치는 16입니다
accumulation_results = []
for batch_size, accumulation_steps in [(16, 1), (4, 4)]:
    accumulation_results.append(measure_train_step(
        model,                                  # 체크포인팅을 켠 QLoRA
        optimizer,                              # 같은 AdamW
        token_ids,                              # 노무 토큰 열
        batch_size=batch_size,                  # 한 번에 넣는 배치 크기
        seq_len=labor_max_length,               # 가장 긴 노무 예시 길이
        accumulation_steps=accumulation_steps,  # 갱신 한 번에 더하는 배치 수
    ))

# 유효 배치와 길이는 두 행이 같아 빼고 봅니다
accumulation_table = pd.DataFrame(accumulation_results)
display(accumulation_table[["배치 크기", "누적 횟수"] + measure_columns])

**결과 읽기**
- **같은 유효 배치 16:** 강사 실행에서 최대 할당 메모리가 배치 16 × 누적 1의 10.26GB에서 배치 4 × 누적 4의 3.15GB로 약 70% 줄고, 갱신당 시간은 0.954초에서 2.403초로 약 2.5배 늘었습니다. 같은 수의 예시를 계산하지만, 작은 배치를 여러 번 돌리면 GPU를 덜 채운 채 계산합니다.

**확인 퀴즈**: 노무 학습 데이터(1,169행)를 배치 16 × 누적 1 대신 배치 4 × 누적 4로 학습하면 에폭당 갱신 횟수는 어떻게 바뀔까요?

<details><summary>정답 보기</summary>

- **바뀌지 않습니다.** 갱신 횟수는 유효 배치로 정해지므로 둘 다 에폭당 74번입니다. 학습률과 갱신 횟수가 같으니 학습 설정을 다시 고르지 않아도 됩니다.
- **바뀌는 것:** 최대 할당 메모리는 배치 4만큼으로 줄고, 갱신 한 번에 작은 배치를 네 번 계산하므로 시간이 늘어납니다.

</details>

### 7-4. 옵티마이저 상태 줄이기

AdamW는 학습 파라미터마다 상태 두 개(1차·2차 이동평균)를 파라미터와 같은 dtype으로 둡니다. QLoRA 어댑터는 bf16이라 상태도 16비트이고, 이 상태를 더 줄이는 옵티마이저가 둘 있습니다([bitsandbytes 옵티마이저 설명](https://huggingface.co/docs/bitsandbytes/main/en/explanations/optimizers)).

- **8비트 옵티마이저(`bitsandbytes.optim.AdamW8bit`):** 상태 두 개를 블록마다 양자화해 8비트로 저장합니다.
- **paged 옵티마이저(`PagedAdamW8bit`):** 상태를 CUDA 통합 메모리(GPU와 CPU가 함께 쓰는 주소 공간)에 둡니다. GPU 메모리가 실제로 모자랄 때만 상태를 CPU 메모리로 내보냈다가, 갱신할 때 다시 가져옵니다.

<img src="images/14_paged_optimizer.png" width="1050" alt="32비트 옵티마이저 상태는 8비트 상태의 4배입니다(QLoRA 어댑터처럼 bf16이면 2배). 8비트 옵티마이저는 이 상태를 줄이고, paged 옵티마이저는 GPU 메모리가 모자랄 때만 상태를 CPU로 옮깁니다.">

In [ ]:
# 어댑터의 학습 파라미터 수(0.8B, r=8 all-linear)
adapter_count = small_counts["어댑터"]

# 상태 두 개(1차, 2차 이동평균)의 바이트: AdamW 는 어댑터와 같은 bf16이라 값마다 2 * 2, 8비트는 1 * 2
optimizer_table = pd.DataFrame([
    {"옵티마이저": "AdamW (bf16 어댑터)", "SFTConfig optim": "adamw_torch_fused", "상태(MB)": adapter_count * 4 / 1024 ** 2,
     "GPU가 모자랄 때": "OOM"},
    {"옵티마이저": "AdamW8bit", "SFTConfig optim": "adamw_8bit", "상태(MB)": adapter_count * 2 / 1024 ** 2,
     "GPU가 모자랄 때": "OOM"},
    {"옵티마이저": "PagedAdamW8bit", "SFTConfig optim": "paged_adamw_8bit", "상태(MB)": adapter_count * 2 / 1024 ** 2,
     "GPU가 모자랄 때": "상태를 CPU로 내보냄"},
])
display(optimizer_table.round(1))

**결과 읽기**
- **어댑터 학습에서는 효과가 작습니다:** 상태가 약 20.6MB에서 약 10.3MB로 줄어도, GB 단위로 측정한 최대 할당 메모리에 비하면 작습니다. 풀 파인튜닝처럼 0.8B 전체(약 7.5억 개)를 fp32로 학습하면 상태가 약 5.6GB에서 약 1.4GB로 줄어 효과가 큽니다.

## 8. 24GB GPU에서 9B 학습 설정 고르기

### 8-1. 9B 가중치 메모리 계산

같은 계열의 `Qwen/Qwen3.5-9B`를 같은 식으로 계산합니다. 가중치는 받지 않고 설정 파일(config.json)만 받아 meta 장치에 모델 뼈대를 만들어 셉니다.

- **meta 장치:** 값 없이 모양만 있는 텐서를 두는 PyTorch 장치라 메모리를 거의 쓰지 않습니다.

In [ ]:
from transformers import AutoConfig

BIG_MODEL_NAME = "Qwen/Qwen3.5-9B"

# 9B: 설정 파일만 받아 meta 장치에 뼈대를 만듭니다. text_config 는 이 모델의 언어 모델 부분 설정입니다
big_config = AutoConfig.from_pretrained(BIG_MODEL_NAME).text_config
with torch.device("meta"):
    big_skeleton = AutoModelForCausalLM.from_config(big_config)
big_counts = count_params(big_skeleton)

# 9B 계산값
big_weight_table = weight_table(big_counts)
display(big_weight_table)

**결과 읽기**
- **4비트 효과가 큽니다:** 9B는 선형 층이 파라미터의 약 77%(0.8B는 약 66%)라, bf16 베이스 약 16.7GB가 4비트에서 약 7.1GB가 됩니다.
- **24GB GPU:** 풀 파인튜닝은 약 133GB라 들어가지 않고, bf16 LoRA는 베이스 가중치만 빼면 약 7.3GB가 남아 로짓과 활성값 자리가 거의 없습니다. QLoRA는 약 16.9GB가 남습니다.

### 8-2. 후보를 측정해 설정 고르기

4비트 9B에 그래디언트 체크포인팅을 켜고, 유효 배치 64를 만드는 후보 3개를 길이 `labor_max_length`로 고릅니다.

- **로짓 하한으로 먼저 거르기:** 4비트 가중치와 로짓 하한의 합이 예산을 넘는 후보는 측정하지 않습니다. 하한만으로 넘으면 실제로는 더 크므로, 측정하면 GPU 메모리를 넘어 OOM으로 멈출 수 있습니다.
- **고르는 규칙:** 남은 후보를 측정해 예약 메모리가 예산 18GB 이하인 후보 가운데 갱신당 시간이 가장 짧은 설정을 고릅니다. 예산은 24GB에서 CUDA 컨텍스트와 메모리 조각을 위한 여유를 남겨 정했습니다.
- **내려받기:** 처음 실행하면 bf16 가중치 파일(비전 부분 포함 약 18GB)을 내려받아 불러올 때 4비트로 바꾸므로, RunPod 디스크에 그 이상 여유가 있어야 합니다. 9B도 어휘가 같아 노무 토큰 열(`token_ids`)을 그대로 씁니다.

In [ ]:
# 0.8B 측정에 쓴 모델과 옵티마이저를 지웁니다
del model, optimizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 9B를 4비트로 불러와 체크포인팅을 켜고 같은 lora_config 어댑터를 붙입니다
# 0.8B에 쓴 lora_config 를 9B에 다시 붙여 base_model_name_or_path 가 바뀌었다는 UserWarning 이 나옵니다. 어댑터를 새로 붙이는 것이라 오류가 아닙니다
model = load_lora_model(
    BIG_MODEL_NAME,               # 9B 모델
    "4bit",                       # QLoRA
    gradient_checkpointing=True,  # 층 활성값을 줄입니다
)
optimizer = torch.optim.AdamW(
    model.parameters(),  # 기울기가 있는 어댑터만 갱신됩니다
    lr=2e-4,             # 0.8B 측정과 같은 학습률
)

# 메모리 예산(GB, 1024**3 바이트)과 4비트 9B 가중치(GB, 계산 테이블의 마지막 행 QLoRA)
big_budget_gb = 18
big_weight_gb = big_weight_table["베이스 가중치(GB)"].iloc[-1]

# 토큰 하나의 fp32 로짓 사본 바이트 = 어휘 수 * 4. 0.8B도 어휘가 같습니다
logit_bytes_per_token = big_config.vocab_size * 4

# 유효 배치 64를 만드는 후보: (배치 크기, 누적 횟수)
big_candidates = [(64, 1), (8, 8), (4, 16)]
big_rows = []
for batch_size, accumulation_steps in big_candidates:
    # 하한(GB) = 가중치 + 배치 크기 * 길이 * 토큰 하나의 로짓 바이트 / 1024 ** 3
    lower_bound_gb = big_weight_gb + batch_size * labor_max_length * logit_bytes_per_token / 1024 ** 3
    row = {"배치 크기": batch_size, "누적 횟수": accumulation_steps, "하한(GB)": round(lower_bound_gb, 1)}

    # 하한이 예산 이하인 후보만 측정합니다. 넘는 후보의 측정 열은 비워 둡니다(NaN)
    if lower_bound_gb <= big_budget_gb:
        result = measure_train_step(
            model,                                  # 체크포인팅을 켠 4비트 9B
            optimizer,                              # AdamW
            token_ids,                              # 노무 토큰 열
            batch_size=batch_size,                  # 후보의 배치 크기
            seq_len=labor_max_length,               # 가장 긴 노무 예시 길이
            accumulation_steps=accumulation_steps,  # 후보의 누적 횟수
        )
        for column in measure_columns:
            row[column] = result[column]
    big_rows.append(row)
big_table = pd.DataFrame(big_rows)
display(big_table)

In [ ]:
# 측정한 후보 가운데 예약 메모리가 예산 이하이고 갱신당 시간이 가장 짧은 후보를 고릅니다
big_chosen = None
for row in big_rows:
    if "예약(GB)" in row and row["예약(GB)"] <= big_budget_gb:
        if big_chosen is None or row["갱신당 시간(초)"] < big_chosen["갱신당 시간(초)"]:
            big_chosen = row
print("고른 배치 크기:", big_chosen["배치 크기"], "| 누적 횟수:", big_chosen["누적 횟수"])

**결과 읽기**
- **하한으로 거른 후보:** 배치 64는 가장 긴 예시 210토큰으로 하한이 약 19.6GB라 예산 18GB를 넘어 측정하지 않았습니다.
- **측정한 후보(강사 실행):**
  - 배치 8 × 누적 8: 최대 할당 12.40GB, 예약 13.79GB, 갱신당 17.027초
  - 배치 4 × 누적 16: 최대 할당 9.85GB, 예약 10.56GB, 갱신당 16.675초
- **고른 설정:** 둘 다 예약 메모리가 예산 18GB 이하라, 규칙대로 갱신당 시간이 더 짧은 배치 4 × 누적 16을 골랐습니다.
- **하한과 실제:** 실제 최대 할당 메모리는 로짓 하한보다 2.0~3.7GB 컸습니다. 하한은 거르는 데만 쓰고, 고르기는 측정으로 합니다.
- **학습할 때:** 고른 배치 크기와 누적 횟수, 길이 `labor_max_length`, 체크포인팅을 9절 테이블의 `SFTConfig` 인자로 옮깁니다. 이 교안은 설정만 고르고 학습하지 않습니다.

In [ ]:
# 9B 모델과 옵티마이저를 지웁니다
del model, optimizer
gc.collect()
torch.cuda.empty_cache()

## 9. 메모리가 부족할 때 손대는 순서

학습 중 `CUDA out of memory`가 나면 대가가 작은 것부터 하나씩 바꾸고, 바꿀 때마다 최대 할당 메모리를 다시 측정합니다.

<img src="images/15_oom_order.png" width="1050" alt="메모리가 부족할 때 손대는 순서: 1 배치 줄이고 기울기 누적, 2 max_length 맞추기, 3 그래디언트 체크포인팅, 4 4비트 베이스, 5 8비트 옵티마이저, 6 더 큰 GPU.">

| 순서 | 바꾸는 것 | `SFTConfig`·불러오기 인자 |
|---|---|---|
| 1 | 배치 줄이고 기울기 누적으로 유효 배치 유지 | `per_device_train_batch_size`, `gradient_accumulation_steps` |
| 2 | `max_length`를 데이터에 맞게(긴 예시를 자르면 그 배치의 활성값이 줄고, 잘린 부분은 학습에서 빠짐) | `max_length` |
| 3 | 그래디언트 체크포인팅 | `gradient_checkpointing=True` |
| 4 | 4비트 베이스(QLoRA) | `from_pretrained(..., quantization_config=bnb_config)` |
| 5 | 8비트 옵티마이저(학습 파라미터가 많을 때 효과가 큼) | `optim="adamw_8bit"` |
| 6 | 더 큰 GPU | - |

순서는 이 교안이 대가가 작은 것부터 정한 것입니다. 관련 문서: [TRL 메모리 줄이기 문서](https://huggingface.co/docs/trl/reducing_memory_usage)

## 10. 🖐️ 함께 따라하기: 전월세 상담 학습 설정 고르기

본실습의 9B 설정 고르기를 지난 시간 함께 따라하기의 전월세 상담 데이터에 적용해, 0.8B QLoRA를 학습할 설정을 측정으로 고릅니다.

**본실습과 다른 점**
- **데이터:** `data/따라하기/lease_train.jsonl`(지난 시간 함께 따라하기와 같은 주택임대차보호법 질문-답변)입니다. 가장 긴 예시의 토큰 수를 길이로 쓰고, 전월세 토큰 열(`lease_token_ids`)로 측정합니다.
- **모델과 후보:** 체크포인팅을 켠 0.8B QLoRA에 유효 배치 32를 만드는 후보 4개(32 × 1, 16 × 2, 8 × 4, 4 × 8)입니다.
- **예산:** 이 학습에 쓸 수 있는 GPU 메모리를 4GB로 정했다고 가정합니다. 하한으로 거르고 예약 메모리로 고르는 규칙은 본실습과 같습니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 빈칸은 후보 측정과 고르는 조건에만 있고, 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

**확인 기준:** `lease_chosen`이 `None`이 아니고, 그 예약 메모리가 예산 4GB 이하입니다.

### 10-1. 길이 정하기와 토큰 열

In [ ]:
# 전월세 학습 데이터의 예시마다 토큰 수를 세고, 같은 문자열의 토큰 번호를 한 줄로 이어 붙입니다
lease_rows = load_jsonl(data_dir / "따라하기" / "lease_train.jsonl")
lease_lengths = []
lease_ids = []
for row in lease_rows:
    chat_text = tokenizer.apply_chat_template(
        row["messages"],  # system, user, assistant 메시지
        tokenize=False,   # 문자열로 받습니다
    )
    row_ids = tokenizer(chat_text)["input_ids"]
    lease_lengths.append(len(row_ids))
    lease_ids.extend(row_ids)

# 가장 긴 예시의 토큰 수를 길이로 씁니다
lease_max_length = max(lease_lengths)
lease_token_ids = torch.tensor(lease_ids)
print("가장 긴 예시(토큰):", lease_max_length)

### 10-2. 하한으로 거르고 후보 측정하기

In [ ]:
# 체크포인팅을 켠 0.8B QLoRA 모델과 옵티마이저
# 9B에 쓴 lora_config 를 0.8B에 다시 붙여 base_model_name_or_path 가 바뀌었다는 UserWarning 이 나옵니다. 오류가 아닙니다
model = load_lora_model(
    MODEL_NAME,                   # 0.8B 모델
    "4bit",                       # QLoRA
    gradient_checkpointing=True,  # 층 활성값을 줄입니다
)
optimizer = torch.optim.AdamW(
    model.parameters(),  # 기울기가 있는 어댑터만 갱신됩니다
    lr=2e-4,             # 본실습과 같은 학습률
)

# 메모리 예산(GB)과 4비트 0.8B 가중치(GB, 4-3 계산 테이블의 마지막 행 QLoRA)
lease_budget_gb = 4
lease_weight_gb = small_weight_table["베이스 가중치(GB)"].iloc[-1]

# 토큰 하나의 로짓 바이트 = 출력층 크기(어휘 수) * 4(fp32)
logit_bytes_per_token = model.get_output_embeddings().out_features * 4

# 유효 배치 32를 만드는 후보: (배치 크기, 누적 횟수)
lease_candidates = [(32, 1), (16, 2), (8, 4), (4, 8)]
lease_rows_measured = []
for batch_size, accumulation_steps in lease_candidates:
    # 하한(GB) = 가중치 + 배치 크기 * 길이 * 토큰 하나의 로짓 바이트 / 1024 ** 3
    lower_bound_gb = lease_weight_gb + batch_size * lease_max_length * logit_bytes_per_token / 1024 ** 3
    row = {"배치 크기": batch_size, "누적 횟수": accumulation_steps, "하한(GB)": round(lower_bound_gb, 1)}

    # 하한이 예산 이하인 후보만 측정합니다
    if lower_bound_gb <= lease_budget_gb:
        # 🖐️ 함께 따라하기: 전월세 토큰 열과 lease_max_length 로 이 후보를 측정하세요(누적 횟수도 넘깁니다)
        result = ...
        for column in measure_columns:
            row[column] = result[column]
    lease_rows_measured.append(row)
lease_table = pd.DataFrame(lease_rows_measured)
display(lease_table)

### 10-3. 예산 안에서 고르기

In [ ]:
# 측정한 후보 가운데 예약 메모리가 예산 이하이고 갱신당 시간이 가장 짧은 후보를 고릅니다
lease_chosen = None
for row in lease_rows_measured:
    if "예약(GB)" in row and row["예약(GB)"] <= lease_budget_gb:
        # 🖐️ 함께 따라하기: 아직 고른 후보가 없거나, 이 후보의 갱신당 시간이 더 짧으면 바꾸세요
        if ...:
            lease_chosen = row
print("고른 배치 크기:", lease_chosen["배치 크기"], "| 누적 횟수:", lease_chosen["누적 횟수"])

**결과 읽기**
- **하한으로 거른 후보:** 배치 32는 가장 긴 예시 121토큰으로 하한이 약 4.3GB라 예산 4GB를 넘어 측정하지 않습니다.
- **측정한 후보:** 강사 실행에서 예약 메모리가 16 × 2는 7.44GB, 8 × 4는 4.12GB로 예산 4GB를 넘어 걸러지고, 2.47GB인 4 × 8을 골랐습니다. 하한으로는 셋 모두 예산 안이었으므로, 하한은 거르기에만 쓰고 고르기는 측정으로 합니다.
- **시간의 대가:** 갱신당 시간이 16 × 2 1.080초, 8 × 4 2.088초, 4 × 8 4.339초로, 고른 4 × 8이 가장 느리고 16 × 2의 약 4배입니다. 예산이 넉넉하면 더 큰 배치로 같은 학습을 더 빨리 끝냅니다.

**확인 퀴즈**: 예산이 더 줄어 측정한 후보가 모두 예산을 넘으면(`lease_chosen`이 None) 무엇부터 바꿀까요?

<details><summary>정답 보기</summary>

- **손대는 순서의 1번부터 봅니다.** 배치를 2나 1로 줄이고 누적을 16이나 32로 늘리면 유효 배치 32를 지키면서 층 활성값과 로짓이 가장 작아집니다. 대가는 갱신당 시간입니다.
- **길이는 이미 가장 긴 예시에 맞췄습니다.** 더 짧게 자르면 긴 답의 뒷부분이 학습에서 빠지므로 1번으로 예산 안에 들면 자르지 않습니다. 체크포인팅과 4비트는 이미 켜져 있고, 8비트 옵티마이저는 어댑터만 학습해 줄어드는 양이 작습니다.

</details>

## 11. 핵심 정리

- **학습 메모리:** 가중치, 기울기, 옵티마이저 상태, 활성값의 네 가지입니다. LoRA와 QLoRA는 학습하는 값이 어댑터뿐이라, 차이는 고정한 베이스 가중치에서 납니다.
- **4비트 저장:** 가중치를 64개 블록마다 absmax로 나눠 NF4 16단계 가운데 가장 가까운 번호로 저장합니다. 이중 양자화는 absmax 몫을 가중치 하나에 0.5비트에서 약 0.127비트로 줄입니다.
- **4비트로 불러오기:** `BitsAndBytesConfig`의 네 인자를 직접 넣습니다(기본값은 FP4, 이중 양자화 끔, float32 계산). 선형 층만 4비트가 되어 0.8B는 bf16 1.4GB -> 4비트 0.71GB로 절반 정도만 줄어듭니다.
- **어댑터 붙이기:** `SFTTrainer`는 베이스를 고정하고 어댑터를 붙여 bf16으로 바꿉니다. `prepare_model_for_kbit_training`은 부르지 않습니다.
- **측정:** 설정마다 갱신 몇 번의 최대 할당 메모리와 갱신당 시간을 비교합니다. 0.8B는 가중치보다 로짓과 활성값이 메모리를 좌우해, 4비트로 아끼는 몫이 작습니다.
- **메모리 줄이기:** 체크포인팅은 층 활성값을 줄이고 시간을 대가로 냅니다. 배치를 줄이고 누적으로 유효 배치를 지키며, `max_length`는 가장 긴 예시로 정합니다. 8비트 옵티마이저는 어댑터 학습에서는 효과가 작습니다.
- **9B를 24GB에:** 4비트로 베이스 가중치가 7.1GB(bf16 16.7GB)가 됩니다. 로짓 하한으로 후보를 거르고 측정해, 예산 18GB 안에서 가장 빠른 배치 4 × 누적 16을 골랐습니다.
- **OOM 순서:** 배치 줄이고 누적 -> `max_length` 맞춤 -> 체크포인팅 -> 4비트 베이스 -> 8비트 옵티마이저 -> 더 큰 GPU.